<a href="https://colab.research.google.com/github/NxrFesdac/bourbaki-nlp-avanzado/blob/main/modulo3/RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faiss-cpu langchain-experimental langchain-huggingface sentence-transformers langchain_openai

In [ ]:
import getpass
from openai import OpenAI
import numpy as np

api_key = getpass.getpass("Introduce tu OpenAI API Key: ")
client = OpenAI(api_key=api_key)

documentos = [
    "La política de GlobalCorp permite 2 días de teletrabajo a la semana.",
    "Las solicitudes se hacen en el portal WorkLife con 48h de antelación.",
    "El periodo de prueba de 3 meses excluye el beneficio de teletrabajo.",
    "El contacto oficial para dudas es rrhh@globalcorp.com."
]

Introduce tu OpenAI API Key: ··········


In [ ]:
pregunta = "¿Cuántos días puedo trabajar desde casa?"

# Unimos los documentos en un solo bloque de contexto
contexto = "\n".join(documentos)

response = client.chat.completions.create(
    model="gpt-4o-mini", # Modelo económico y potente
    messages=[
        {"role": "system", "content": "Eres un asistente de RRHH de GlobalCorp. Responde solo basándote en el contexto."},
        {"role": "user", "content": f"Contexto:\n{contexto}\n\nPregunta: {pregunta}"}
    ]
)

print(response.choices[0].message.content)

Puedes trabajar desde casa 2 días a la semana, siguiendo la política de GlobalCorp.


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def get_embedding(text):
    return client.embeddings.create(input=[text], model="text-embedding-3-small").data[0].embedding

# 1. Convertir los 4 documentos a vectores
embeddings_docs = [get_embedding(doc) for doc in documentos]

# 2. Convertir la pregunta
pregunta = "¿A quién contacto si tengo problemas?"
embedding_pregunta = get_embedding(pregunta)

# 3. Calcular similitud de coseno
scores = cosine_similarity([embedding_pregunta], embeddings_docs)[0]
top_idx = np.argmax(scores)

print(f"Documento más relevante: {documentos[top_idx]}")
print(f"Puntaje de similitud: {scores[top_idx]:.4f}")

Documento más relevante: El contacto oficial para dudas es rrhh@globalcorp.com.
Puntaje de similitud: 0.4800


In [ ]:
import faiss

# 1. text-embedding-3-small tiene una dimensión de 1536, definimos la vectorDB a esa dimensión
dimension = 1536
index = faiss.IndexFlatL2(dimension)

# 2. Agregar documentos al índice
# Ya tenemos los embeddings_docs del paso anterior
index.add(np.array(embeddings_docs).astype('float32'))


query_usuario = "Necesito teletrabajo pero llevo solo 1 mes en la empresa."

# A. Retrieval (Recuperación)
query_vec = np.array([get_embedding(query_usuario)]).astype('float32')
D, I = index.search(query_vec, k=1) # Buscamos el mejor resultado, normalmente RAG usa de 1 - 10 en promedio, depende del use case.
contexto_recuperado = documentos[I[0][0]]

# B. Augmentation & Generation (Aumentar y Generar)
final_response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "system", "content": "Responde dudas de empleados usando el contexto."},
        {"role": "user", "content": f"Contexto: {contexto_recuperado}\n\nPregunta: {query_usuario}"}
    ]
)

print(f"Contexto usado: {contexto_recuperado}")
print(f"Respuesta IA: {final_response.choices[0].message.content}")

Contexto usado: El periodo de prueba de 3 meses excluye el beneficio de teletrabajo.
Respuesta IA: Según el contexto, durante el periodo de prueba de 3 meses no se aplica el beneficio de teletrabajo. Por lo tanto, al llevar solo 1 mes en la empresa, no puedes acceder aún a la modalidad de teletrabajo.


In [ ]:
# Tomamos el documento sobre el periodo de prueba (Documento #3)
texto_largo = documentos[2]

def chunk_text(text, size=25, overlap=5):
    chunks = []
    for i in range(0, len(text), size - overlap):
        chunks.append(text[i : i + size])
    return chunks

fragmentos = chunk_text(texto_largo)
print(f"Texto original: {texto_largo}")
for i, f in enumerate(fragmentos):
    print(f"Chunk {i}: '{f}'")

Texto original: El periodo de prueba de 3 meses excluye el beneficio de teletrabajo.
Chunk 0: 'El periodo de prueba de 3'
Chunk 1: ' de 3 meses excluye el be'
Chunk 2: 'el beneficio de teletraba'
Chunk 3: 'trabajo.'


In [ ]:
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt_tab')

contexto = "\n".join(documentos)

# This safely splits by sentence, ignoring abbreviations
chunks = sent_tokenize(contexto)

print(chunks)


['La política de GlobalCorp permite 2 días de teletrabajo a la semana.', 'Las solicitudes se hacen en el portal WorkLife con 48h de antelación.', 'El periodo de prueba de 3 meses excluye el beneficio de teletrabajo.', 'El contacto oficial para dudas es rrhh@globalcorp.com.']


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
def semantic_chunker(docs, embeddings, threshold=0.4):
    if not docs: return []

    # Start the first chunk with the first document
    chunks = [[docs[0]]]

    # Compare each doc to the one right before it
    for i in range(1, len(docs)):
        sim = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]

        # If similar, append to the current chunk's list. If not, start a new list.
        if sim >= threshold:
            chunks[-1].append(docs[i])
        else:
            chunks.append([docs[i]])

    # Join the grouped sentences back together into strings
    return [" ".join(chunk) for chunk in chunks]

print(semantic_chunker(documentos, embeddings_docs))

['La política de GlobalCorp permite 2 días de teletrabajo a la semana. Las solicitudes se hacen en el portal WorkLife con 48h de antelación. El periodo de prueba de 3 meses excluye el beneficio de teletrabajo.', 'El contacto oficial para dudas es rrhh@globalcorp.com.']
